# Learning the round / elongated tagging

Can the measurements reproduce the round / elongated tags given by eye (`annotate.py`)? Three models of increasing
complexity are trained on the tagged cells and tested **leave-one-series-out**: each stack is predicted by a model
that never saw it, so the scores say how well the model would tag a new stack.

1. **threshold** on one feature (`THRESHOLD_FEATURE`), chosen on the training stacks;
2. **logistic regression** on `FEATURES` (interpretable: one weight per feature);
3. **random forest** on `FEATURES` (non-linear).

The chosen model then predicts every kept cell, including the real-but-dirty cells that have no round / elongated
tag, and lists tagged cells where it confidently disagrees, for a second look with `annotate.py --amend`.
Results go to `results/predicted_phenotypes.csv`, which `phenotype_differences.ipynb` can use.

## Settings

In [ ]:
from pathlib import Path

FEATURES = ["axis_ratio_2_1", "body_elongation", "body_flatness", "body_volume_um3", "process_fraction",
            "n_dendrites", "mean_dendrite_reach_um", "max_dendrite_reach_um",
            "dendrite_starriness", "dendrite_planarity", "dendrite_randomness", "dendrite_axis_alignment", "Solidity"]
THRESHOLD_FEATURE = "axis_ratio_2_1"   # the single-feature baseline
MODEL = "logistic"                     # model used for the predictions: threshold, logistic or forest
DISAGREE_PROBABILITY = 0.75            # tagged cells the model contradicts with at least this confidence are listed
RESULTS = Path("results")
SEED = 20261014

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, cohen_kappa_score, roc_auc_score
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from osteo_common import FEATURE_LABELS, GROUP_COLOURS, INK, add_features, clean, load, style

style()
RESULTS.mkdir(exist_ok=True)
rng = np.random.default_rng(SEED)

## Data

Junk cells are removed; the training set is every kept cell tagged elongated or round.

In [ ]:
all_cells, dendrites = load()
cells, summary = clean(all_cells)
cells = add_features(cells, dendrites)
train = cells[cells["phenotype"] != ""].reset_index(drop=True)
X, y = train[FEATURES].to_numpy(float), (train["phenotype"] == "round").to_numpy()
groups = train["series"].to_numpy()
print(f"{len(cells)} kept cells; {len(train)} tagged ({y.sum()} round, {(~y).sum()} elongated); "
      f"{(cells['phenotype'] == '').sum()} without a round/elongated tag to predict")
train.groupby("series")["phenotype"].value_counts().unstack(fill_value=0)

## Leave-one-series-out comparison

**Balanced accuracy** averages the hit rate of the two classes (0.5 = chance, whatever the class sizes);
**Cohen's κ** is agreement beyond chance (0 none, 0.4–0.6 moderate, 0.6–0.8 substantial, > 0.8 almost perfect;
Landis & Koch, *Biometrics* 33:159, 1977, [doi:10.2307/2529310](https://doi.org/10.2307/2529310)); **AUC** measures
how well the probabilities rank round above elongated, independent of the cut.

In [ ]:
def threshold_model(train_x, train_y):
    # best cut on one feature by balanced accuracy, round = above the cut
    v = train_x[:, FEATURES.index(THRESHOLD_FEATURE)]
    cuts = np.unique(v[np.isfinite(v)])
    best = max(cuts, key=lambda t: balanced_accuracy_score(train_y, v >= t))
    return lambda x: (x[:, FEATURES.index(THRESHOLD_FEATURE)] >= best).astype(float), best


MODELS = {
    "logistic": lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                      LogisticRegression(class_weight="balanced", max_iter=1000)),
    "forest": lambda: make_pipeline(SimpleImputer(strategy="median"),
                                    RandomForestClassifier(500, min_samples_leaf=3, class_weight="balanced_subsample",
                                                           random_state=SEED, n_jobs=-1)),
}
cv_prob = {name: np.full(len(y), np.nan) for name in ["threshold", *MODELS]}
cuts = []
for tr, te in LeaveOneGroupOut().split(X, y, groups):
    predict, cut = threshold_model(X[tr], y[tr])
    cv_prob["threshold"][te] = predict(X[te])
    cuts.append(cut)
    for name, make in MODELS.items():
        cv_prob[name][te] = make().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]

rows = []
for name, p in cv_prob.items():
    pred = p >= 0.5
    rows.append(dict(model=name, balanced_accuracy=balanced_accuracy_score(y, pred), kappa=cohen_kappa_score(y, pred),
                     AUC=roc_auc_score(y, p) if name != "threshold" else np.nan,
                     round_found=f"{(pred & y).sum()}/{y.sum()}", elongated_found=f"{(~pred & ~y).sum()}/{(~y).sum()}"))
scores = pd.DataFrame(rows).set_index("model")
print(f"threshold on {THRESHOLD_FEATURE}: cut {np.median(cuts):.2f} (range over folds {min(cuts):.2f}–{max(cuts):.2f})")
scores.round(3)

In [ ]:
per_series = pd.DataFrame({name: pd.Series((p >= 0.5) == y).groupby(groups).mean() for name, p in cv_prob.items()})
per_series["cells"] = pd.Series(y).groupby(groups).size()
per_series.round(2)

## What the models use

Logistic weights are per standard deviation of each feature (positive = more round); forest importance is the drop in balanced accuracy when a feature is shuffled.

In [ ]:
logistic = MODELS["logistic"]().fit(X, y)
forest = MODELS["forest"]().fit(X, y)
imp = permutation_importance(forest, X, y, scoring="balanced_accuracy", n_repeats=20, random_state=SEED)
weights = pd.DataFrame({"feature": [FEATURE_LABELS.get(f, f) for f in FEATURES],
                        "logistic weight": logistic[-1].coef_[0], "forest importance": imp.importances_mean})
weights = weights.reindex(weights["logistic weight"].abs().sort_values(ascending=False).index)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, col in zip(axes, ["logistic weight", "forest importance"]):
    ax.barh(weights["feature"], weights[col], color=[GROUP_COLOURS["round"] if v > 0 else GROUP_COLOURS["elongated"]
                                                      for v in weights[col]], height=0.6)
    ax.axvline(0, color=INK["context"], linewidth=1)
    ax.set(title=col, xlabel="← elongated    round →" if col == "logistic weight" else "")
axes[0].invert_yaxis()
fig.tight_layout()
fig.savefig(RESULTS / "classifier_features.png")
weights.round(3)

## Predictions

The chosen `MODEL`, fitted on all tagged cells, predicts every kept cell. For tagged cells the leave-one-series-out probability is kept as well (`p_round_cv`), which is what the disagreement list uses.

In [ ]:
Xall = cells[FEATURES].to_numpy(float)
if MODEL == "threshold":
    predict, cut = threshold_model(X, y)
    p_all = predict(Xall)
else:
    p_all = (logistic if MODEL == "logistic" else forest).predict_proba(Xall)[:, 1]
cells["p_round"] = p_all
cells["predicted"] = np.where(p_all >= 0.5, "round", "elongated")
cv = pd.Series(cv_prob[MODEL], index=pd.MultiIndex.from_frame(train[["series", "label"]]))
cells["p_round_cv"] = cv.reindex(pd.MultiIndex.from_frame(cells[["series", "label"]])).to_numpy()

out = cells[["series", "label", "tag", "review", "phenotype", "p_round", "p_round_cv", "predicted"]]
out.to_csv(RESULTS / "predicted_phenotypes.csv", index=False)
untagged = cells[cells["phenotype"] == ""]
print(f"wrote {RESULTS / 'predicted_phenotypes.csv'}; cells without a round/elongated tag predicted "
      f"{(untagged['predicted'] == 'round').sum()} round, {(untagged['predicted'] == 'elongated').sum()} elongated")

t = cells[cells["phenotype"] != ""]
confident = np.maximum(t["p_round_cv"], 1 - t["p_round_cv"]) >= DISAGREE_PROBABILITY
wrong = t[confident & ((t["p_round_cv"] >= 0.5) != (t["phenotype"] == "round"))]
print(f"{len(wrong)} tagged cells the model contradicts with >= {DISAGREE_PROBABILITY:.0%} confidence "
      "(check with: python3 annotate.py --amend --only <tag>):")
wrong[["series", "label", "phenotype", "p_round_cv", THRESHOLD_FEATURE, "body_elongation", "dendrite_starriness"]].round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for name, colour in GROUP_COLOURS.items():
    m = train["phenotype"] == name
    ax.scatter(train.loc[m, THRESHOLD_FEATURE], cv_prob[MODEL][m.to_numpy()], s=26, color=colour, edgecolor="white",
               linewidth=0.7, label=f"tagged {name}")
ax.axhline(0.5, color=INK["muted"], linestyle="--", linewidth=1)
ax.set(xlabel=FEATURE_LABELS.get(THRESHOLD_FEATURE, THRESHOLD_FEATURE), ylabel=f"P(round), {MODEL}, unseen stack",
       title="Leave-one-series-out probability against the tag")
ax.legend(loc="upper left")
fig.tight_layout()
fig.savefig(RESULTS / "classifier_probabilities.png")